In [0]:
# step 1: load one ses file from the delta table into pandas
import pandas as pd

#target_file    = "AthenaMoCC_MoCC_33484-413637-413637-20260209-1285820_1_30262199175_PATIENT_20260211005448.xml"   #athena
#target_file    = "SES_13d77c4b-490a-44eb-a99e-98dc9c46743a.xml"   #ses
target_file    =  "EPIC_8295510_202502190330524718647.xml"  #epic

target_section = "Results"

# derive vendor name from the file name 
if target_file.lower().startswith("athena"):
    vendor_name = "athena"
else:
    vendor_name = target_file.split("_")[0].lower()

view_name = vendor_name + "_" + target_section.lower()


df = spark.sql(f"""
    select medical_record_filename,coalesce(variant_get(CCDA_RAW,'$.id._root', 'string'),variant_get(CCDA_RAW,'$.id._extension', 'string'))as document_id
    , CCDA_RAW
    from racd_prod.shared.brnz_ccda_raw_variant
    where medical_record_filename = '{target_file}'
    limit 1
""")
document_id    = df.select("document_id").collect()[0][0]

print("rows loaded :", df.count())
print("file        :", df.select("medical_record_filename").collect()[0][0])
print("document_id :", document_id)
print(vendor_name)

rows loaded : 1
file        : EPIC_8295510_202502190330524718647.xml
document_id : 1.2.840.114350.1.13.424.2.7.8.688883.1122351808
epic


In [0]:
# step 2: variant columns need to be cast to string first
# this converts the variant structure to a plain json string
# so python can work with it in the next step

import json

raw_value = df.select("ccda_raw").collect()[0][0]

# variant comes back as a string already serialised as json
# if it is not a string cast it explicitly
if not isinstance(raw_value, str):
    raw_value = str(raw_value)

json_text = raw_value

print("type         :", type(json_text))
print("preview      :", json_text[:500])


type         : <class 'str'>
preview      : {"_xmlns":"urn:hl7-org:v3","author":{"assignedAuthor":{"addr":{"_nullFlavor":"NA"},"assignedAuthoringDevice":{"manufacturerModelName":"Epic - Version 10.9","softwareName":"Epic - Version 10.9"},"id":{"_extension":10.9,"_root":"1.2.840.114350.1.1"},"representedOrganization":{"addr":{"_use":"WP","city":"SYLVANIA","country":"US","county":"LUCAS","postalCode":43560,"state":"OH","streetAddressLine":"5855 Monroe St"},"id":{"_extension":33900,"_root":"1.2.840.114350.1.13.424.2.7.2.688879"},"name":"ProM


In [0]:
# step 3: parse the json string into a python dictionary
# json.loads gives us a native python dict to walk through

data = json.loads(json_text)

print("top level keys:", list(data.keys()))

top level keys: ['_xmlns', 'author', 'code', 'component', 'componentOf', 'confidentialityCode', 'custodian', 'documentationOf', 'effectiveTime', 'id', 'languageCode', 'legalAuthenticator', 'participant', 'realmCode', 'recordTarget', 'relatedDocument', 'setId', 'templateId', 'title', 'typeId', 'versionNumber']


In [0]:
# step 4: navigate the json structure to find the target section
# in ccda json the structure follows:
# component -> structuredBody -> component (array) -> section -> title
# variant_get(col, '$.section.title', 'string') = 'Results'

results_section = None

def find_section_in_json(node, section_title):
    # handle lists -- recurse into each item
    if isinstance(node, list):
        for item in node:
            result = find_section_in_json(item, section_title)
            if result is not None:
                return result

    # handle dicts -- look for a section key with a matching title
    if isinstance(node, dict):
        if "section" in node:
            section = node["section"]
            if isinstance(section, dict):
                title = section.get("title", "")
                if isinstance(title, str) and section_title.lower() in title.lower():
                    return section

        # keep searching deeper in all values
        for value in node.values():
            result = find_section_in_json(value, section_title)
            if result is not None:
                return result

    return None

results_section = find_section_in_json(data, target_section)

if results_section:
    print("section found :", results_section.get("title"))
    print("top level keys:", list(results_section.keys()))
else:
    print("section not found - check target_section value")

section found : Results
top level keys: ['code', 'entry', 'id', 'templateId', 'text', 'title']


In [0]:
# step 5: loop through the json section recursively and capture all columns
# in json/variant the attributes are just keys in the same dict
# they are prefixed with _ like _root, _code, _value, _displayName
# so we split the dict into two parts:
# attributes = keys starting with _  (equivalent to xml attributes)
# children   = keys not starting with _ (equivalent to xml child elements)

import json

rows = []

def build_row(tag, path, depth, node_type, position, child_count, text_value, all_attrs):
    # this mirrors the xml rows.append block exactly
    # all_attrs here is the dict of _ prefixed keys from the json dict
    return {
        "medical_record_filename" : target_file,
        "document_id"             : document_id,
        "vendor_name"             : vendor_name,
        "section_name"            : target_section,
        "path"                    : path,
        "tag"                     : tag,
        "depth"                   : depth,
        "node_type"               : node_type,
        "is_array"                : position >= 0,
        "child_count"             : child_count,
        "text_value"              : str(text_value).strip()[:200] if text_value is not None else None,
        # full attributes as json string -- equivalent to xml all_attrs dump
        "attributes"              : json.dumps(all_attrs) if all_attrs else None,
        # individual attribute columns -- in json these come from _ prefixed keys
        # e.g. {"_root": "2.16", "_code": "30954-2"} maps to attr_root, attr_code etc
        "attr_root"               : all_attrs.get("_root"),
        "attr_extension"          : all_attrs.get("_extension"),
        "attr_code"               : all_attrs.get("_code"),
        "attr_code_system"        : all_attrs.get("_codeSystem"),
        "attr_code_system_name"   : all_attrs.get("_codeSystemName"),
        "attr_display_name"       : all_attrs.get("_displayName"),
        "attr_value"              : str(all_attrs.get("_value")) if all_attrs.get("_value") is not None else None,
        "attr_unit"               : all_attrs.get("_unit"), # unit tag not found in results section
        "attr_null_flavor"        : all_attrs.get("_nullFlavor"),
        "attr_xsi_type"           : all_attrs.get("_xsi:type"),
    }

def walk(node, depth, parent_path, tag, position):

    # build path with array index if this is inside a list
    if position >= 0:
        path = parent_path + "/" + tag + "[" + str(position) + "]"
    else:
        path = parent_path + "/" + tag

    if isinstance(node, dict):
        # split the dict into attributes (_ prefixed keys) and child elements
        # attributes: {"_root": "2.16", "_code": "30954-2"} -- same as xml attrs
        # children  : {"code": {...}, "title": "Results"}   -- same as xml child tags
        all_attrs  = {k: v for k, v in node.items() if k.startswith("_")}
        children   = {k: v for k, v in node.items() if not k.startswith("_")}

        # check if any child value is a list -- means this struct contains arrays
        has_array_children = any(isinstance(v, list) for v in children.values())

        if position >= 0 and has_array_children:
            node_type = "array of structs (contains arrays)"
        elif position >= 0:
            node_type = "array of structs"
        elif has_array_children:
            node_type = "struct (contains arrays)"
        else:
            node_type = "struct"

        # text value for a dict comes from a # text key if present
        text_value = node.get("#text")

        rows.append(build_row(
            tag         = tag,
            path        = path,
            depth       = depth,
            node_type   = node_type,
            position    = position,
            child_count = len(children),
            text_value  = text_value,
            all_attrs   = all_attrs,
        ))

        # recurse into child elements only (not _ prefixed keys)
        for child_key, child_value in children.items():
            if child_key == "#text":
                # skip -- already captured as text_value above
                continue
            elif isinstance(child_value, list):
                # list = array -- walk each item with its index
                for i, item in enumerate(child_value):
                    walk(item, depth + 1, path, child_key, i)
            else:
                walk(child_value, depth + 1, path, child_key, -1)

    elif isinstance(node, list):
        # bare list -- recurse each item with index
        for i, item in enumerate(node):
            walk(item, depth + 1, parent_path, tag, i)

    else:
        # leaf node -- plain string, number or bool
        node_type = "array of leaves" if position >= 0 else "leaf"

        rows.append(build_row(
            tag         = tag,
            path        = path,
            depth       = depth,
            node_type   = node_type,
            position    = position,
            child_count = 0,
            text_value  = node,
            all_attrs   = {},     
        ))

# kick off the walk from the results section
for key, value in results_section.items():
    if isinstance(value, list):
        for i, item in enumerate(value):
            walk(item, depth=1, parent_path="/section", tag=key, position=i)
    else:
        walk(value, depth=1, parent_path="/section", tag=key, position=-1)

df_nodes = pd.DataFrame(rows)

print("total nodes  :", len(df_nodes))
print("total leaves :", len(df_nodes[df_nodes["node_type"].str.startswith("leaf")]))
print("\nnode type breakdown:")
print(df_nodes["node_type"].value_counts())

total nodes  : 37373
total leaves : 4778

node type breakdown:
node_type
struct                                16990
array of structs                       6339
array of leaves                        6224
leaf                                   4778
struct (contains arrays)               2225
array of structs (contains arrays)      817
Name: count, dtype: int64


In [0]:
%skip
# step 6: filter to leaf nodes only
# leaves are where the actual data values live
# this summary shows what fields exist, how often they appear
# and a sample value so the we can understand what each field contains

df_leaves = df_nodes[df_nodes["node_type"] == "leaf"].copy()

summary = (
    df_leaves
    .groupby(["tag", "path", "depth"])
    .agg(
        occurrences  = ("tag",         "count"),
        filled_count = ("text_value",  lambda x: x.notna().sum()),
        sample_value = ("text_value",  lambda x: x.dropna().iloc[0] if x.notna().any() else None),
        sample_code  = ("attr_code",   lambda x: x.dropna().iloc[0] if x.notna().any() else None),
        sample_oid   = ("attr_root",   lambda x: x.dropna().iloc[0] if x.notna().any() else None)
    )
    .reset_index()
    .sort_values(["depth", "occurrences"], ascending=[True, False])
)

print("leaf node count :", len(summary))
display(summary)

In [0]:
%skip
# step 7: create a temporary view from the pandas dataframe
spark_df = spark.createDataFrame(df_nodes.astype(str))
spark_df.createOrReplaceTempView("vw_ccda_json_profile")

print("rows in view           :", spark_df.count())

In [0]:
    
# step 7: write to a delta table with merge logic
# use try/except to check if the table exists

for col in df_nodes.select_dtypes(include=['object']).columns:
    df_nodes[col] = df_nodes[col].astype(str)

from delta.tables import DeltaTable
delta_table_name = "lab_teams.racd.ccda_json_profile"


spark_df = spark.createDataFrame(df_nodes)

try:
    # try if table does not exist this will fail
    delta_table = DeltaTable.forName(spark, delta_table_name)
    
    print("delta table exists, running merge")
    
    (
        delta_table.alias("existing")
        .merge(
            spark_df.alias("new"),
            """
            existing.medical_record_filename = new.medical_record_filename
            and existing.section_name        = new.section_name
            and existing.path                = new.path
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    
    print("merge complete")

except Exception:
    # if table does not exist it create one first
    print("delta table does not exist, creating on first run")
    
    (
        spark_df.write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .saveAsTable(delta_table_name)
    )
    
    print("delta table created :", delta_table_name)

# confirm row counts
total_rows = spark.table(delta_table_name).count()
print("total rows in table  :", total_rows)

delta table exists, running merge
merge complete
total rows in table  : 38818


###Analysis

In [0]:
%sql
select * from
--distinct vendor_name, path from 
--vw_ccda_json_profile
lab_teams.racd.ccda_json_profile 
--where 
--path like '/section/entry[2]/component[20]/observation'
--path rlike '^/section/entry(\\[\\d+\\])?/organizer/component(\\[\\d+\\])?/observation/entryRelationship/act'
--path rlike '^/section/entry(\\[\\d+\\])/organizer/specimen/specimenRole/id(\\[\\d+\\])'
-- path like '/section/code'
 --tag = 'specimen'
--AND tag in ('effectiveTime','value', 'low', 'high')
--and vendor_name='epic'
order by path limit 100


medical_record_filename,document_id,vendor_name,section_name,path,tag,depth,node_type,is_array,child_count,text_value,attributes,attr_root,attr_extension,attr_code,attr_code_system,attr_code_system_name,attr_display_name,attr_value,attr_unit,attr_null_flavor,attr_xsi_type
EPIC_8295510_202502190330524718647.xml,1.2.840.114350.1.13.424.2.7.8.688883.1122351808,epic,Results,/section/code,code,1,struct,false,0,None,"{""_code"": ""30954-2"", ""_codeSystem"": ""2.16.840.1.113883.6.1"", ""_codeSystemName"": ""LOINC"", ""_displayName"": ""Relevant diagnostic tests/laboratory data Narrative""}",None,None,30954-2,2.16.840.1.113883.6.1,LOINC,Relevant diagnostic tests/laboratory data Narrative,None,None,None,None
AthenaMoCC_MoCC_33484-413637-413637-20260209-1285820_1_30262199175_PATIENT_20260211005448.xml,48891c02-0707-11f1-890e-9d7cef3fe925,athena,Results,/section/code,code,1,struct,false,0,None,"{""_code"": ""30954-2"", ""_codeSystem"": ""2.16.840.1.113883.6.1"", ""_codeSystemName"": ""LOINC""}",None,None,30954-2,2.16.840.1.113883.6.1,LOINC,None,None,None,None,None
EPIC_8295510_202502190330524718647.xml,1.2.840.114350.1.13.424.2.7.8.688883.1122351808,epic,Results,/section/entry[0],entry,1,array of structs,true,1,None,"{""_typeCode"": ""DRIV""}",None,None,None,None,None,None,None,None,None,None
AthenaMoCC_MoCC_33484-413637-413637-20260209-1285820_1_30262199175_PATIENT_20260211005448.xml,48891c02-0707-11f1-890e-9d7cef3fe925,athena,Results,/section/entry[0],entry,1,array of structs,true,1,None,None,None,None,None,None,None,None,None,None,None,None
EPIC_8295510_202502190330524718647.xml,1.2.840.114350.1.13.424.2.7.8.688883.1122351808,epic,Results,/section/entry[0]/organizer,organizer,2,struct (contains arrays),false,12,None,"{""_classCode"": ""BATTERY"", ""_moodCode"": ""EVN""}",None,None,None,None,None,None,None,None,None,None
AthenaMoCC_MoCC_33484-413637-413637-20260209-1285820_1_30262199175_PATIENT_20260211005448.xml,48891c02-0707-11f1-890e-9d7cef3fe925,athena,Results,/section/entry[0]/organizer,organizer,2,struct (contains arrays),false,6,None,"{""_classCode"": ""BATTERY"", ""_moodCode"": ""EVN""}",None,None,None,None,None,None,None,None,None,None
EPIC_8295510_202502190330524718647.xml,1.2.840.114350.1.13.424.2.7.8.688883.1122351808,epic,Results,/section/entry[0]/organizer/author,author,3,struct (contains arrays),false,3,None,None,None,None,None,None,None,None,None,None,None,None
EPIC_8295510_202502190330524718647.xml,1.2.840.114350.1.13.424.2.7.8.688883.1122351808,epic,Results,/section/entry[0]/organizer/author/assignedAuthor,assignedAuthor,4,struct,false,4,None,None,None,None,None,None,None,None,None,None,None,None
EPIC_8295510_202502190330524718647.xml,1.2.840.114350.1.13.424.2.7.8.688883.1122351808,epic,Results,/section/entry[0]/organizer/author/assignedAuthor/addr,addr,5,struct,false,0,None,"{""_nullFlavor"": ""UNK""}",None,None,None,None,None,None,None,None,UNK,None
EPIC_8295510_202502190330524718647.xml,1.2.840.114350.1.13.424.2.7.8.688883.1122351808,epic,Results,/section/entry[0]/organizer/author/assignedAuthor/id,id,5,struct,false,0,None,"{""_nullFlavor"": ""UNK"", ""_root"": ""2.16.840.1.113883.4.6""}",2.16.840.1.113883.4.6,None,None,None,None,None,None,None,UNK,None


#### Reading mapping table and append the data into stage

***Loading mapping csv file to a table***

In [0]:
mapping_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/lab_teams/racd/ccda_mapping/observation_mapping.csv")

mapping_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("lab_teams.racd.ccda_source_mapping_tm_v1")

print("mapping table created")
mapping_df.show(truncate=False)

mapping table created
+-------------+-----------+-----------+------------+------------------------------------+---------------------------------------------------------------------------------------------------+-----------+---------+
|source_system|source_type|vendor_name|section_name|column_name                         |column_path                                                                                        |entity     |is_active|
+-------------+-----------+-----------+------------+------------------------------------+---------------------------------------------------------------------------------------------------+-----------+---------+
|CCDA         |file       |epic       |Results     |OBSERVATION_ID_ROOT                 |/section/entry/organizer/component/observation/id/root                                             |OBSERVATION|Y        |
|CCDA         |file       |epic       |Results     |OBSERVATION_ID_EXT                  |/section/entry/organizer/component/observ

***Build the column list dynamically from the mapping table***

In [0]:
mapping_df = spark.sql("""
    select column_name, column_path
    from lab_teams.racd.ccda_source_mapping_tm_v1
    where is_active = 'Y'
    and   entity    = 'OBSERVATION'
    order by column_name
""").toPandas()

case_statements = []

for _, row in mapping_df.iterrows():
    col_name = row["column_name"]
    col_path = row["column_path"]

    last_segment = col_path.split("/")[-1]

    attr_map = {
        "root"          : "attr_root",
        "extension"     : "attr_extension",          
        "code"           : "attr_code",
        "codeSystemName" : "attr_code_system_name",
        "displayName"    : "attr_display_name",
        "value"          : "attr_value",
        "unit"           : "attr_unit",
        "nullFlavor"     : "attr_null_flavor",
        "xsiType"        : "attr_xsi_type"
    }

    attr_col = attr_map.get(last_segment, "text_value")

    case_statements.append(
        f"    max(case when column_name = '{row['column_name']}' "  
        f"then {attr_col} end) as {col_name}"                       
    )

pivot_cols = ",\n".join(case_statements)

print("dynamic columns built :", len(case_statements))
print(pivot_cols)

dynamic columns built : 50
    max(case when column_name = 'DEVICE_ID' then attr_root end) as DEVICE_ID,
    max(case when column_name = 'ENCOUNTER_ID_ALT' then text_value end) as ENCOUNTER_ID_ALT,
    max(case when column_name = 'ENCOUNTER_ID_ROOT' then attr_root end) as ENCOUNTER_ID_ROOT,
    max(case when column_name = 'IMMUNIZATION_ID' then attr_root end) as IMMUNIZATION_ID,
    max(case when column_name = 'MEDICATION_REQUEST_ID' then attr_root end) as MEDICATION_REQUEST_ID,
    max(case when column_name = 'OBSERVATION_AUTHOR_REFERENCE_CODE' then attr_root end) as OBSERVATION_AUTHOR_REFERENCE_CODE,
    max(case when column_name = 'OBSERVATION_AUTHOR_REFERENCE_NOTE_TEXT' then text_value end) as OBSERVATION_AUTHOR_REFERENCE_NOTE_TEXT,
    max(case when column_name = 'OBSERVATION_AUTHOR_REFERENCE_NOTE_TMSTP' then attr_value end) as OBSERVATION_AUTHOR_REFERENCE_NOTE_TMSTP,
    max(case when column_name = 'OBSERVATION_CATEGORY_CODE' then attr_code end) as OBSERVATION_CATEGORY_CODE,
    

**Build and execute the full dynamic pivot query**

In [0]:
query = f"""
select
    medical_record_filename,
    vendor_name,
    document_id,
{pivot_cols} # dynamic columns
from (
    select
        p.medical_record_filename,
        p.vendor_name,
        p.document_id,
        m.column_name,
        regexp_extract(
            p.path,
            '(.*observation)',
            1
        ) as observation_instance,
        p.attr_root,
        p.attr_extension,
        p.attr_code,
        p.attr_code_system_name,
        p.attr_display_name,
        p.attr_value,
        p.attr_unit,
        p.attr_null_flavor,
        p.attr_xsi_type,
        p.text_value
    from lab_teams.racd.ccda_source_mapping_tm_v1 m
    left join lab_teams.racd.ccda_json_profile p
        on  regexp_replace(p.path, '\\\\[\\\\d+\\\\]', '')
            =
            regexp_replace(m.column_path, '/[^/]+$', '')
        and p.vendor_name  = m.vendor_name
        and p.section_name = m.section_name
    where m.is_active = 'Y'
    and   m.entity    = 'OBSERVATION'
) base
group by
    medical_record_filename,
    vendor_name,
    document_id,
    observation_instance
order by
    medical_record_filename
"""

print("executing query :")
print(query)

df_observation = spark.sql(query)
display(df_observation)

executing query :

select
    medical_record_filename,
    vendor_name,
    document_id,
    max(case when column_name = 'DEVICE_ID' then attr_root end) as DEVICE_ID,
    max(case when column_name = 'ENCOUNTER_ID_ALT' then text_value end) as ENCOUNTER_ID_ALT,
    max(case when column_name = 'ENCOUNTER_ID_ROOT' then attr_root end) as ENCOUNTER_ID_ROOT,
    max(case when column_name = 'IMMUNIZATION_ID' then attr_root end) as IMMUNIZATION_ID,
    max(case when column_name = 'MEDICATION_REQUEST_ID' then attr_root end) as MEDICATION_REQUEST_ID,
    max(case when column_name = 'OBSERVATION_AUTHOR_REFERENCE_CODE' then attr_root end) as OBSERVATION_AUTHOR_REFERENCE_CODE,
    max(case when column_name = 'OBSERVATION_AUTHOR_REFERENCE_NOTE_TEXT' then text_value end) as OBSERVATION_AUTHOR_REFERENCE_NOTE_TEXT,
    max(case when column_name = 'OBSERVATION_AUTHOR_REFERENCE_NOTE_TMSTP' then attr_value end) as OBSERVATION_AUTHOR_REFERENCE_NOTE_TMSTP,
    max(case when column_name = 'OBSERVATION_CATEGOR

medical_record_filename,vendor_name,document_id,DEVICE_ID,ENCOUNTER_ID_ALT,ENCOUNTER_ID_ROOT,IMMUNIZATION_ID,MEDICATION_REQUEST_ID,OBSERVATION_AUTHOR_REFERENCE_CODE,OBSERVATION_AUTHOR_REFERENCE_NOTE_TEXT,OBSERVATION_AUTHOR_REFERENCE_NOTE_TMSTP,OBSERVATION_CATEGORY_CODE,OBSERVATION_CATEGORY_CODE_DISPLAY_NAME,OBSERVATION_CATEGORY_CODE_SYSTEM,OBSERVATION_CATEGORY_CODE_SYSTEM_NAME,OBSERVATION_CODE,OBSERVATION_CODE_NULL_FLAVOR,OBSERVATION_CODE_SYSTEM,OBSERVATION_CODE_SYSTEM_DISPLAY_NAME,OBSERVATION_CODE_SYSTEM_NAME,OBSERVATION_CODE_TEXT,OBSERVATION_EFFECTIVE_TIME_HIGH_VALUE,OBSERVATION_EFFECTIVE_TIME_LOW_VALUE,OBSERVATION_EFFECTIVE_TIME_VALUE,OBSERVATION_ID_EXT,OBSERVATION_ID_ROOT,OBSERVATION_INTERPRETATION_CODE,OBSERVATION_INTERPRETATION_CODE_DISPLAY_NAME,OBSERVATION_INTERPRETATION_CODE_SYSTEM,OBSERVATION_INTERPRETATION_CODE_TEXT,OBSERVATION_METHOD_CODE,OBSERVATION_METHOD_CODE_DISPLAY_NAME,OBSERVATION_METHOD_CODE_SYSTEM_NAME,OBSERVATION_METHOD_CODE_TEXT,OBSERVATION_REFERENCE_RANGE_HIGH_QTY,OBSERVATION_REFERENCE_RANGE_HIGH_QTY_CODE,OBSERVATION_REFERENCE_RANGE_LOW_QTY,OBSERVATION_REFERENCE_RANGE_LOW_QTY_CODE,OBSERVATION_REFERENCE_RANGE_TEXT,OBSERVATION_STATUS_CODE,OBSERVATION_VALUE,OBSERVATION_VALUE_CODE,OBSERVATION_VALUE_CODE_SYSTEM,OBSERVATION_VALUE_CODE_SYSTEM_NAME,OBSERVATION_VALUE_NULL_FLAVOR,OBSERVATION_VALUE_TRANSLATION_NULL_FLAVOR,OBSERVATION_VALUE_UNIT,OBSERVATION_VALUE_XSI_TYPE,PRACTITIONER_ID,PRACTITIONER_ROLE_ID,PROVENANCE_ID,SERVICE_REQUEST_ID,SPECIMEN_ID
null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
EPIC_8295510_202502190330524718647.xml,epic,1.2.840.114350.1.13.424.2.7.8.688883.1122351808,null,null,null,null,null,null,null,null,null,null,null,null,None,UNK,None,None,None,null,null,null,20241230201025+0000,424468452,1.2.840.114350.1.13.424.2.7.2.798268,null,null,null,null,null,null,null,null,null,null,null,null,null,completed,None,7,None,Epic.Result.Type,None,null,None,None,null,null,null,null,null
EPIC_8295510_202502190330524718647.xml,epic,1.2.840.114350.1.13.424.2.7.8.688883.1122351808,null,null,null,null,null,null,null,null,null,null,null,null,718-7,None,None,None,LOINC,None,null,null,20241226120819+0000,424393772.3,1.2.840.114350.1.13.424.2.7.6.798268.2000,L,None,None,Low,null,null,null,null,null,null,null,null,null,completed,9.6,None,None,None,None,null,g/dL,None,null,null,null,null,null
EPIC_8295510_202502190330524718647.xml,epic,1.2.840.114350.1.13.424.2.7.8.688883.1122351808,null,null,null,null,null,null,null,null,null,null,null,null,None,UNK,None,None,None,null,null,null,20241222141629+0000,424066317,1.2.840.114350.1.13.424.2.7.2.798268,null,null,null,null,null,null,null,null,null,null,null,null,null,completed,None,16,None,Epic.Result.Type,None,null,None,None,null,null,null,null,null
EPIC_8295510_202502190330524718647.xml,epic,1.2.840.114350.1.13.424.2.7.8.688883.1122351808,null,null,null,null,null,null,null,null,null,null,null,null,None,UNK,None,None,None,null,null,null,20241227030456+0000,424486233,1.2.840.114350.1.13.424.2.7.2.798268,null,null,null,null,null,null,null,null,null,null,null,null,null,completed,None,7,None,Epic.Result.Type,None,null,None,None,null,null,null,null,null
EPIC_8295510_202502190330524718647.xml,epic,1.2.840.114350.1.13.424.2.7.8.688883.1122351808,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,None,None,None,None,150 - 450 X10E9/L,null,null,null,null,null,null,null,null,null,null,null,null,null,null
EPIC_8295510_202502190330524718647.xml,epic,1.2.840.114350.1.13.424.2.7.8.688883.1122351808,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,109,mmol/L,98,mmol/L,

***Append data to a table***
--TBD